<a href="https://colab.research.google.com/github/ThomasAlbin/Space-Science-With-Python/blob/main/2026/04_SPICE_Apohis_2029_Flyby.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SPICE Lecture 04: Apophis — The 2029 Earth Flyby

On **13 April 2029**, asteroid **99942 Apophis** will fly past Earth at a distance of roughly 38 000 km — closer than geostationary satellites. For a brief window it will pass inside Earth's **sphere of influence (SOI)**, making this one of the most closely watched planetary encounters in modern astronomy.

In this notebook we use SPICE to answer three questions:

1. *When* does Apophis enter and leave Earth's SOI?
2. How do its heliocentric orbital elements change as a result of the flyby?
3. What is the exact moment and distance of closest approach?

We use the small-body SPK kernel `sb-99942-218.bsp` released by JPL/NAIF, which provides Apophis's predicted trajectory through 2029. Apophis carries the NAIF ID **`2099942`**.

Earth's sphere of influence radius follows the same formula used for Saturn in the previous notebook:

$$ r_{\mathrm{SOI}} = a \left( \frac{m_{\mathrm{Earth}}}{m_{\mathrm{Sun}}} \right)^{2/5} $$

where $a$ is Earth's semi-major axis (~1 AU).

In [1]:
import datetime
from pathlib import Path
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import spiceypy as spice

plt.style.use('dark_background')
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['font.family'] = 'STIXGeneral'
plt.rcParams['font.size'] = 11

In [2]:
data_dir = Path("../data")
data_dir.mkdir(exist_ok=True)

kernels = {
    # Generic kernels
    "naif0012.tls": "https://naif.jpl.nasa.gov/pub/naif/generic_kernels/lsk/naif0012.tls",
    "pck00010.tpc": "https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/pck00010.tpc",
    "gm_de440.tpc": "https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/gm_de440.tpc",
    "de432s.bsp": "https://naif.jpl.nasa.gov/pub/naif/generic_kernels/spk/planets/de432s.bsp",
    # Apophis kernel
    "sb-99942-218.bsp": "https://naif.jpl.nasa.gov/pub/naif/ORX/kernels/spk/sb-99942-218.bsp"
}

for name, url in kernels.items():
    path = data_dir / name
    if not path.exists():
        print(f"Downloading {name}...")
        urllib.request.urlretrieve(url, path)
    else:
        print(f"{name} already exists.")

for name in kernels:
    spice.furnsh(str(data_dir / name))

naif0012.tls already exists.
pck00010.tpc already exists.
gm_de440.tpc already exists.
de432s.bsp already exists.
sb-99942-218.bsp already exists.


---
## 1. Earth's Sphere of Influence

We extract the gravitational parameters of the Sun and Earth from the loaded PCK kernel and compute the radius of Earth's SOI. Earth orbits at roughly 1 AU, so we use `spice.convrt()` to get that distance in km directly rather than querying a state vector.

In [3]:
one_au = spice.convrt(1.0, 'AU', 'KM')

gm_sun = spice.bodvrd("SUN", 'GM', 1)[1][0]   # km^3/s^2
gm_earth = spice.bodvrd("EARTH BARYCENTER", 'GM', 1)[1][0]  # km^3/s^2

print(f"GM of the Sun:    {gm_sun:.5e} km^3/s^2")
print(f"GM of Saturn:     {gm_earth:.5e} km^3/s^2")
print(f"Mass ratio (Saturn / Sun): {gm_earth / gm_sun:.6e}")

# ── Sphere of Influence: r = a * (m_saturn / m_sun) ** (2/5) ──────────────────
soi_radius_km = one_au * (gm_earth / gm_sun) ** (2.0 / 5.0)

print(f"\nEarth's Sphere of Influence radius: {soi_radius_km:.3f} km "
      f"({soi_radius_km / one_au:.4f} AU)")

GM of the Sun:    1.32712e+11 km^3/s^2
GM of Saturn:     4.03503e+05 km^3/s^2
Mass ratio (Saturn / Sun): 3.040433e-06

Earth's Sphere of Influence radius: 929179.387 km (0.0062 AU)


---
## 2. When Does Apophis Enter Earth's SOI?

We use the SPICE geometry finder `gfposc` to search the full year 2029 for intervals where Apophis's distance from the **Earth Barycenter** is **smaller than** the SOI radius. We use a 1-hour step size — much finer than the 1-day step used for the Cassini search — because the entire flyby lasts only a few days.

In [4]:
et_start = spice.str2et('2029-01-01')
et_end   = spice.str2et('2029-12-31')

# Define the search window (confinement window)
cnfine = spice.utils.support_types.SPICEDOUBLE_CELL(2)
spice.wninsd(et_start, et_end, cnfine)

# Geometric "finder" settings and requirements
target = "2099942"
frame  = "ECLIPJ2000"
abcorr = "NONE"
obsrvr = "EARTH BARYCENTER"

crdsys = "SPHERICAL"
coord  = "RADIUS"

relate = "<"
refval = soi_radius_km
adjust = 0.0
step   = 3600.0

# Result window
result = spice.utils.support_types.SPICEDOUBLE_CELL(1000)

# Actual geometry finder
spice.gfposc(target,
             frame,
             abcorr,
             obsrvr,
             crdsys,
             coord,
             relate,
             refval,
             adjust,
             step,
             1000,
             cnfine,
             result)

count = spice.wncard(result)
print(f"Found {count} interval(s) where Apophis is inside Earth's SOI.\n")
for i in range(count):
    interval_start, interval_end = spice.wnfetd(result, i)
    print(f"  Entered SOI: {spice.et2utc(interval_start, 'C', 3)}")
    print(f"  Window ends: {spice.et2utc(interval_end,   'C', 3)}  (= end of search window / kernel coverage)")

Found 1 interval(s) where Apophis is inside Earth's SOI.

  Entered SOI: 2029 APR 12 03:17:43.602
  Window ends: 2029 APR 15 16:35:43.516  (= end of search window / kernel coverage)


---
## 3. Orbital Elements Before and After the Flyby

A close planetary encounter exchanges energy and angular momentum between the asteroid and the planet, permanently altering the asteroid's heliocentric orbit. We compute the **osculating orbital elements** of Apophis at the moment it enters and exits Earth's SOI using `spice.oscelt()` with the Sun's GM, then compare key elements side by side.

In [5]:
# Let's compute the orbital elements of Apophis in sun-centric coordinates before and after the flyby
state_before_flyby, _ = spice.spkgeo(targ=2099942,
                                     et=interval_start,
                                     ref="ECLIPJ2000",
                                     obs=0)

state_after_flyby, _ = spice.spkgeo(targ=2099942,
                                    et=interval_end,
                                    ref="ECLIPJ2000",
                                    obs=0)

conics_before = spice.oscelt(state=state_before_flyby,
                             et=interval_start,
                             mu=gm_sun)

conics_after = spice.oscelt(state=state_after_flyby,
                            et=interval_end,
                            mu=gm_sun)

In [6]:
rp_before, ecc_before, inc_before, lnode_before, argp_before, _, _, _ = conics_before
rp_after, ecc_after, inc_after, lnode_after, argp_after, _, _, _ = conics_after

print(f"Perihelion before flyby: {rp_before / one_au:.2f}")
print(f"Perihelion after flyby: {rp_after / one_au:.2f}\n")
print(f"Eccentricity before flyby: {np.degrees(ecc_before):.2f}")
print(f"Eccentricity after flyby: {np.degrees(ecc_after):.2f}\n")
print(f"Inclination before flyby: {np.degrees(inc_before):.2f}")
print(f"Inclination after flyby: {np.degrees(inc_after):.2f}\n")

Perihelion before flyby: 0.74
Perihelion after flyby: 0.89

Eccentricity before flyby: 11.07
Eccentricity after flyby: 10.89

Inclination before flyby: 3.38
Inclination after flyby: 2.25



---
## 4. Closest Approach

Knowing *when* Apophis is inside Earth's SOI is not the same as knowing *when* it is closest. We re-run `gfposc` with `relate = "ABSMIN"` to pin down the single epoch at which Apophis reaches its minimum distance from Earth during 2029.

In [7]:
# Closest Approach
et_start = spice.str2et('2029-01-01')
et_end   = spice.str2et('2029-12-31')

# Define the search window (confinement window)
cnfine = spice.utils.support_types.SPICEDOUBLE_CELL(2)
spice.wninsd(et_start, et_end, cnfine)

# Geometric "finder" settings and requirements
target = "2099942"
frame  = "ECLIPJ2000"
abcorr = "NONE"
obsrvr = "EARTH BARYCENTER"

crdsys = "SPHERICAL"
coord  = "RADIUS"

relate = "ABSMIN"
refval = 0
adjust = 0
step   = 3600.0

# Result window
result = spice.utils.support_types.SPICEDOUBLE_CELL(1000)

# Actual geometry finder
spice.gfposc(target,
             frame,
             abcorr,
             obsrvr,
             crdsys,
             coord,
             relate,
             refval,
             adjust,
             step,
             1000,
             cnfine,
             result)

count = spice.wncard(result)
print(f"Found {count} interval(s) where Apophis is inside Earth's SOI.\n")
for i in range(count):
    flyby_et, _ = spice.wnfetd(result, i)
    print(f"Closest flyby at: {spice.et2utc(flyby_et, 'C', 3)}")

Found 1 interval(s) where Apophis is inside Earth's SOI.

Closest flyby at: 2029 APR 13 21:58:42.587


In [8]:
# What's the distance?
state_flyby, _ = spice.spkgeo(targ=2099942,
                              et=flyby_et,
                              ref="J2000",
                              obs=3)

print(f"Closest distance in km: {np.linalg.norm(state_flyby[:3]):.0f}")

Closest distance in km: 38441


---
## 5. Sanity Check

With the closest-approach epoch in hand we queried the state vector directly via `spice.spkgeo()` and now we also cross-check using the osculating conic at that the SOI entry moment via `spice.oscelt()`. Both methods should give (almost) the same periapsis distance — a useful sanity check that the two SPICE functions are consistent.

In [9]:
# There is another way!
soi_entry_state, _ = spice.spkgeo(targ=2099942,
                                  et=interval_start,
                                  ref="J2000",
                                  obs=3)

conics_flyby = spice.oscelt(state=soi_entry_state,
                            et=interval_start,
                            mu=gm_earth)

print(f"Closest distance, based on conics in km: {conics_flyby[0]:.0f}")

Closest distance, based on conics in km: 39219


There is a slight deviation, since the SPK kernels are more detailed and precise than a simplified 2-body problem. However, in the order of magnitude it works nicely.